# Cross Validation
## The Flaw of the Train/Test Split

Up until now, we have been using train_test_split to divide our data (e.g., 80% for training, 20% for testing).

But what if you get lucky? What if all the easy-to-predict data accidentally ends up in your 20% testing set? Your model will report a 99% accuracy, but it will fail miserably in the real world.

Conversely, what if you get unlucky and all the weirdest, most chaotic outliers end up in your testing set? Your model might actually be great, but it reports a terrible score.

Relying on a single Train/Test split leaves your evaluation up to chance.

## The Solution: K-Fold Cross Validation

K-Fold Cross Validation eliminates the luck factor by testing the model on every single piece of data. Here is how it works:

The Split: Instead of splitting the data into two uneven pieces, you chop your entire dataset into 'K' equal-sized chunks (called "folds"). The industry standard is usually K = 5 or K = 10.

1. The Rotation: * Round 1: You hold out Fold 1 to use as the Test set. You train the model on Folds 2, 3, 4, and 5. You record the accuracy.

2. Round 2: You hold out Fold 2 as the Test set. You train the model on Folds 1, 3, 4, and 5. Record the accuracy.

   Round 3-5: You repeat this until every single fold has had a turn being the Test set.

3. The Final Score: You now have 5 different accuracy scores. You take the average of all 5 scores to get your final, true accuracy.

__(Analogy: Instead of giving a student one final exam that might accidentally have all their favorite questions, you give them 5 different exams over the semester and average the grades. It proves true mastery!)__

https://colab.research.google.com/drive/10A252M7Yoct4lYbXvJrSt5sERQmEQPgp#scrollTo=ycEdB34pxQpA


In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv('https://gist.githubusercontent.com/curran/a08a1080b88344b0c8a7/raw/639388c2cbc2120a14dcf466e85730eb8be498bb/iris.csv')

In [3]:
data.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [11]:
features = data.iloc[:,:-1].values
label = data.iloc[:,-1].values

In [16]:
from sklearn.preprocessing import LabelEncoder

# Convert string labels to integers
le = LabelEncoder()
label = le.fit_transform(label)


In [17]:
label

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [5]:
#List of algos for Classification we learned till date
# 1. LogisticRegression
# 2. KNeighborsClassifier
# 3. DecisionTreeClassifier
# 4. RandomForestClassifier
# 5. BaggingClassifier with LogisticRegression
# 6. BaggingClassifier with KNeighborsClassifier
# 7. BaggingClassifier with SVC
# 8. SVC
# 9. XGBClassifier
# 10. XGBRFClassifier

In [22]:
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

model = LogisticRegression()

scores = cross_val_score(model,
                features,
                label,
                cv=5)

In [23]:
# Benchmark Score
print("Minimum Threshold score (CL) : ", scores.mean())
print("Suggested SL value for the given dataset is ", 1 - scores.mean())

Minimum Threshold score (CL) :  0.9733333333333334
Suggested SL value for the given dataset is  0.026666666666666616


In [24]:
print("Possible Optimal Score is ", scores.max())

Possible Optimal Score is  1.0


In [27]:
# 3. To extract the best training sample that may provide the optimal score

CL = scores.mean()
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()

from sklearn.model_selection import KFold
kfold = KFold(n_splits=5, shuffle=True, random_state=1)
tracker=0

for train, test in kfold.split(features):
    tracker += 1
    X_train, X_test = features[train], features[test]
    y_train, y_test = label[train], label[test]

    model.fit(X_train, y_train)

    if model.score(X_test,y_test) >= CL:
        print("Test Score {} Train Score {} for Sample Split {}".format(model.score(X_test,y_test),model.score(X_train,y_train),tracker))




Test Score 1.0 Train Score 0.9666666666666667 for Sample Split 5


In [28]:
#Extract Best Sample

#Step1: Initialize Algo
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()

#Step2: Initialize K-Fold Cross Validation
from sklearn.model_selection import KFold
kfold = KFold(n_splits=5, #This value MUST be equal to the cv value of cross_val_score
              shuffle=True,
              random_state=1) #This random state is to help me reproduce my same output

#Step3: Initialize tracker to track the best sample

tracker=0

#here split function returns the row index location of the dataset
for train,test in kfold.split(features):
  tracker +=1
  if tracker == 5:
    X_train,X_test,y_train,y_test=features[train],features[test],label[train],label[test]

In [29]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier

results = []
models = [{'model_name': 'LogisticRegression', 'instance': LogisticRegression()},{'model_name': 'KNeighborsClassifier', 'instance': KNeighborsClassifier()},
          {'model_name': 'DecisionTreeClassifier', 'instance':DecisionTreeClassifier()},{'model_name': 'SVC', 'instance': SVC()}, {'model_name': 'Bagging(LogisticRegression)', 'instance':  BaggingClassifier(LogisticRegression())},
          {'model_name': 'Bagging(KNeighborsClassifier)', 'instance': BaggingClassifier(KNeighborsClassifier())}, ]
cvs = [5, 10]
for model in models:
    for cvs_number in cvs:
        scores = cross_val_score(model['instance'],
                             features,
                             label,
                             cv=cvs_number)
        model_name = model['model_name']
        cvs_str = f'cvs: {str(cvs_number)}'
        results.append({'Model': model_name, 'CVS': cvs_str, 'Scores': scores.mean()})

results_df = pd.DataFrame(results)

        # print(f'model name: {model["model_name"]}, with cv of {cvs_number},  scores: {scores.mean()}')
print(results_df)
#%%
best_model = results_df.groupby('Model')['Scores'].mean().idxmax()
best_cvs = results_df[results_df['Model'] == best_model]['CVS'].values[0]

print(f'the best model is: {best_model} with a cvs of: {best_cvs}')

                            Model      CVS    Scores
0              LogisticRegression   cvs: 5  0.973333
1              LogisticRegression  cvs: 10  0.973333
2            KNeighborsClassifier   cvs: 5  0.973333
3            KNeighborsClassifier  cvs: 10  0.966667
4          DecisionTreeClassifier   cvs: 5  0.966667
5          DecisionTreeClassifier  cvs: 10  0.960000
6                             SVC   cvs: 5  0.966667
7                             SVC  cvs: 10  0.973333
8     Bagging(LogisticRegression)   cvs: 5  0.966667
9     Bagging(LogisticRegression)  cvs: 10  0.973333
10  Bagging(KNeighborsClassifier)   cvs: 5  0.973333
11  Bagging(KNeighborsClassifier)  cvs: 10  0.966667
the best model is: LogisticRegression with a cvs of: cvs: 5


In [30]:
results_df

,Model,CVS,Scores
0,LogisticRegression,cvs: 5,0.973333
1,LogisticRegression,cvs: 10,0.973333
2,KNeighborsClassifier,cvs: 5,0.973333
3,KNeighborsClassifier,cvs: 10,0.966667
4,DecisionTreeClassifier,cvs: 5,0.966667
5,DecisionTreeClassifier,cvs: 10,0.960000
6,SVC,cvs: 5,0.966667
7,SVC,cvs: 10,0.973333
8,Bagging(LogisticRegression),cvs: 5,0.966667
9,Bagging(LogisticRegression),cvs: 10,0.973333
